In [1]:
# # TB India v5 -- Simulation (PRODUCTION, HYBRID v5)

# Revised from hybrid v4 after deciding TB Duration = 4.0 yr was implausibly long.
# Hybrid v5 uses a more defensible TB Duration of 3.5 yr (within Tiemersma 2011
# range 2-5 yr), and compensates the prevalence gap by also bumping β slightly.

# ## Hybrid v5 parameters

# | Param | Hybrid v3 | Hybrid v4 (rejected) | **Hybrid v5** | Why |
# |---|---:|---:|---:|---|
# | Base_Infectivity_Constant   | 0.042 | 0.042 | **0.050** | Modest bump to lift inc+prev jointly |
# | TB Duration (yr)            | 2.743 | 4.0   | **3.5**   | Defensible range (Tiemersma 2011) |
# | Death Fraction              | 0.41  | 0.55  | **0.55**  | Lift mortality |
# | Care Seeking Slow Days      | 60    | 60    | 60        | unchanged (~HQ 22d / LQ 37d effective) |
# | TB Presymptomatic Years     | 0.264 | 0.264 | 0.264     | unchanged |

# ## Expected outcome (steady-state approximation)

# Compared to hybrid v3 (inc 150, prev 150, mort 15):

# - **Incidence**: β × (1+ TB_dur lift) ≈ 0.050/0.042 × 3.5/2.74 ≈ 1.52x -> ~190 (target 188 ✓)
# - **Prevalence**: same factor ~1.52x of inc × duration -> ~290 (target 304; should be close)
# - **Mortality**: ratio 0.55/0.41 × 1.52 -> ~30 (target 33; should be close)

# ## Literature defensibility (for methods text)

# - TB Duration 3.5 yr: at the upper end of Tiemersma 2011's reported 2-5 yr range for
#   untreated pulmonary TB; consistent with the literature.
# - Death Fraction 0.55: WHO TB Report case-fatality estimates for India range 0.35-0.60 depending
#   on smear status; 0.55 is within the upper-mid range.
# - β = 0.050: phenomenological per-contact transmission constant, calibrated to WHO targets.

# ## If still off after this run

# - Inc too high & prev OK -> pull β back to 0.045
# - Prev too low but inc OK -> bump TB Duration to 3.75
# - Mort off -> adjust Death Fraction only (range 0.45-0.65)
# - All three jointly too high/low -> bump/pull β alone


## 0. Setup & Imports

Core Python libraries, EMOD/IDM platform modules, and local project imports (, , ).

In [2]:
import sys, os, json, copy, pathlib
import numpy as np
from functools import partial

# emodpy ecosystem
from emodpy import emod_task
from emodpy.emod_task import EMODTask
from emodpy_tbhiv.interventions import purge_campaign_event
import emod_tbhiv.bootstrap as dtk
import emod_api.campaign as camp
import emodpy
import emod_api

print(f'Python:   {sys.version}')
print(f'emodpy:   {emodpy.__version__}')
print(f'emod-api: {emod_api.__version__}')

C:\Users\young\anaconda3\envs\tb_india\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Python:   3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
emodpy:   1.35.1
emod-api: 1.35.1


In [3]:
# Local project modules
import params
import set_config
import manifest
from add_tb_drug_type import add_tb_drug_type, add_tb_drug

# Schema path — must be set before any intervention imports
camp.schema_path = manifest.schema_file
print('Schema loaded:', manifest.schema_file)

Schema loaded: download/schema.json


## 1. Simulation Settings

User-configurable run parameters: experiment name, number of replicates, sweep ranges, and calibration mode toggle.

In [4]:
# ── User settings (PRODUCTION: trend-calibrated params, 2026-09) ──
initial_pop    = 1000
replicates     = 25    # wave-2: 100 points x 25 reps (25 lifts the burden emulators past R2 0.9)
calibration_on = False

# Scenario to run:
#   'base'      — single baseline scenario
#   'acf'       — ACF coverage sweep
#   'cascade'   — cascade efficiency sweep
#   'combined'  — 8 intervention scenarios (production)
#   'wave1_cal' — Wave-1 calibration: 100 LHS design × 10 reps (Baseline only)
#   'wave2_cal' — Wave-2 calibration: 100 refined-region LHS × 25 reps
SCENARIO = 'wave2_cal'   # round-2 wave 2, weighted toward a loose cascade

# v12 calibration: design CSV path (used when SCENARIO == 'wave1_cal' or 'wave2_cal')
CAL_DESIGN_CSV = 'helpers/wave2_design_trend2.csv'   # 70% of points in the loose-cascade stratum
# previous 8-param design: 'helpers/wave1_design.csv'    # v19: refined viable-region wave-2 design    # v7: full 8-scenario sweep

# WHO End TB 2035 targets: 90% incidence reduction, 95% mortality reduction (vs 2015)

# ACF coverage levels (legacy)
ACF_COVERAGE_LEVELS = [0.0, 0.10, 0.20, 0.30, 0.50, 0.70]
CASCADE_EFFICIENCY_LEVELS = [1.0, 0.75, 0.50, 0.25]

# v7 — 8 intervention scenarios per Table 3 of manuscript
# Tuple: (acf_coverage, dx_boost, tx_boost, label)
#   acf_coverage:  0.0 = baseline,  0.5 = ACF active
#   dx_boost:      False = baseline Low-QoC GeneXpert sens (0.65/0.60)
#                  True  = Low-QoC raised to High-QoC sens (0.75/0.65)
#   tx_boost:      False = baseline Low-QoC treatment_fraction 0.55
#                  True  = Low-QoC treatment_fraction 0.85 (digital DOT)
COMBINED_SCENARIOS = [
    (0.0, False, False, "Baseline"),
    (0.5, False, False, "ACF"),
    (0.0, True,  False, "Dx"),
    (0.0, False, True,  "Tx"),
    (0.5, True,  False, "ACF_Dx"),
    (0.5, False, True,  "ACF_Tx"),
    (0.0, True,  True,  "Dx_Tx"),
    (0.5, True,  True,  "ACF_Dx_Tx"),
]

print(f'Scenario: {SCENARIO}  |  Replicates: {replicates}  |  Pop: {initial_pop}')
# v13: SCENARIO-aware total-sim count
if SCENARIO in ('wave1_cal', 'wave2_cal'):
    import pandas as _pd
    _design_n = len(_pd.read_csv(CAL_DESIGN_CSV))
    print(f'Total sims: {_design_n} design points x {replicates} reps = {_design_n * replicates}')
elif SCENARIO == 'combined':
    print(f'Total sims: {len(COMBINED_SCENARIOS)} scenarios x {replicates} reps = {len(COMBINED_SCENARIOS) * replicates}')
elif SCENARIO == 'base':
    print(f'Total sims: 1 scenario x {replicates} reps = {replicates}')
else:
    print(f'Total sims: (depends on sweep — see Cell 24)')


Scenario: wave2_cal  |  Replicates: 25  |  Pop: 1000
Total sims: 100 design points x 25 reps = 2500


## 2. Download Model

Download and register the EMOD executable (Eradication binary) from the IDM repository.

In [5]:
dtk.setup(pathlib.Path(manifest.eradication_path).parent)
print('EMOD binaries ready.')

EMOD binaries ready.


## 3. Platform

Initialise the COMPS cloud platform connection for remote simulation submission and output retrieval.

In [6]:
# ── COMPS login (run this BEFORE the Platform cell below) ──────────────────
# pyCOMPS defaults to a TK popup, which JupyterLab usually cannot display, and
# falls back to a console prompt that goes to the terminal that launched Jupyter
# instead of this cell — which is why no login prompt appears.  Asking for
# ConsoleCredentialPrompt explicitly puts the username/password prompt in this
# cell's output.  Auth tokens expire, so this is needed once per kernel session.
from COMPS import Client
from COMPS.CredentialPrompt import ConsoleCredentialPrompt

COMPS_HOST = 'https://comps.idmod.org'

try:
    _user = Client.auth_manager()._username
    print(f'Already logged in to COMPS as {_user}')
except RuntimeError:
    Client.login(COMPS_HOST, ConsoleCredentialPrompt())
    print(f'Logged in to COMPS as {Client.auth_manager()._username}')


Logged in to COMPS as yjo


In [7]:
from idmtools.core.platform_factory import Platform
platform = Platform('CALCULON', priority='AboveNormal', node_group='idm_48cores')
print('Connected to CALCULON.')


Initializing COMPSPlatform with:
{
   "endpoint": "https://comps.idmod.org",
   "environment": "Calculon",
   "priority": "AboveNormal",
   "node_group": "idm_48cores"
}
Connected to CALCULON.


## 4. Config Builder

Base configuration callback () and helper functions that set TB/HIV disease parameters, simulation duration, and natural-history distributions.

In [8]:
def set_param_fn(config):
    import params as _params  # explicit import to avoid kernel state issues
    config = set_config.set_config(config)
    config.parameters.TB_Enable_Exogenous = 1
    config.parameters.Enable_Strain_Tracking = 1
    config.parameters.Death_Rate_Dependence = "NONDISEASE_MORTALITY_BY_YEAR_AND_AGE_FOR_EACH_GENDER"
    config.parameters.Base_Mortality = 3
    config.parameters.x_Birth = 0.75
    config.parameters.Enable_Default_Reporting = 1
    config.parameters.Spatial_Output_Channels = [
        "Prevalence", "Population", "Active_TB_Prevalence", "New_Active_TB_Infections"
    ]
    config.parameters.Enable_Spatial_Output = 1
    config.parameters.TB_MDR_Fitness_Multiplier = 1.0
    config.parameters.Report_Event_Recorder_Ignore_Events_In_List = 1
    config.parameters.Report_Event_Recorder_Events = [
        'Births', 'EveryUpdate', 'GaveBirth', 'HappyBirthday',
        'NonDiseaseDeaths', 'Pregnant', 'Emigrating', 'Immigrating', 'PropertyChange'
    ]
    # Run to 2035
    config.parameters.Simulation_Duration = (
        _params.burn_initial + _params.burn_predots + (38.0 + 15) * 365.0
    )
    config.parameters.x_Base_Population = 20
    config.parameters.Base_Individual_Sample_Rate = 0.05
    config.parameters.TB_Fast_Progressor_Rate = 0.0175
    return config

In [9]:
def set_duration_no_hiv(task, death_fraction, duration):
    """Convert from years to days, set cure/mortality rates."""
    duration *= 365.0
    loc_rate_recover = (1.0 - death_fraction) / duration
    loc_rate_mort = death_fraction / (1.0 - death_fraction) * loc_rate_recover
    task.config.parameters.TB_Active_Cure_Rate = loc_rate_recover
    task.config.parameters.TB_Active_Mortality_Rate = loc_rate_mort
    return {'TB_Active_Cure_Rate': loc_rate_recover,
            'TB_Active_Mortality_Rate': loc_rate_mort}


def update_more_config(task):
    set_duration_no_hiv(task=task, death_fraction=0.7, duration=1.5)  # v8: shorter natural duration to match P/I ratio
    # v10 Fix A: mortality multiplier moved into map_sample_to_model_input
    # (the multiplier here was being overwritten by subsample sweep).
    task.config.parameters.TB_Presymptomatic_Rate = 0.4
    task.config.parameters.Enable_Property_Output = 1
    return task


## 5. Campaign Builder — Full Diagnostic Cascade

Restores the complete care-seeking / diagnostic / treatment pathway:
- **HSB** for care initiation (with testing-loop logic)
- **DiagnosticTreatNeg** for smear-based diagnosis with explicit negative/default branches
- **RampDTN** for time-varying GeneXpert sensitivity ramp-up
- **ResistanceDiagnostic** for two-step MDR confirmation
- **TBHIVDrugTreatment** for all drug regimens across 3 epochs
- **ACF** (Active Case Finding) — parameterized, added only when 
- **cascade_efficiency** scales care-seeking delay (lower = faster = less health-system delay)

In [10]:
def build_camp(probability_per_step_slow=0.0208, probability_per_step_fast=0.05247,
               personal_hc_budget=7, acf_coverage=0.0, cascade_efficiency=1.0,
               dx_boost=False, tx_boost=False,
               sens_smear_pos_GXL_override=None,
               treatment_fraction_LQ_override=None,
               tf_gain_per_year=0.0, cs_gain_per_year=0.0,
               infectivity_decline_per_year=0.0):
    """
    Build campaign with FULL diagnostic cascade.

    Args:
        probability_per_step_slow: Daily prob of seeking care (slow seekers)
        probability_per_step_fast: Daily prob of seeking care (fast seekers)
        personal_hc_budget: Cash per infection for economic tracking
        acf_coverage: Fraction of subclinical TB detected via ACF (0 = no ACF)
        cascade_efficiency: Multiplier on care-seeking delays (1.0 = baseline)
    """
    import emod_api.interventions.outbreak as ob
    import emod_api.interventions.common as common
    import emodpy_tbhiv.interventions.hsb as hsb
    import emodpy_tbhiv.interventions.art as art
    import emodpy_tbhiv.interventions.diag_treat_neg as dtn
    import emodpy_tbhiv.interventions.tbhiv_treat as treat
    import emodpy_tbhiv.interventions.ramp_dtn as ramp
    import emodpy_tbhiv.interventions.resist_diag as rd
    import emodpy_tbhiv.interventions.hiv_diag as hd
    import emodpy_tbhiv.interventions.hiv_seeding as hs
    import emodpy_tbhiv.interventions.cd4diag as cd4
    import emodpy_tbhiv.interventions.cash as cash

    from helpers.campaign_trend import (cascade_schedule, careseeking_schedule,
                                        infectivity_multiplier_map)

    common.old_adhoc_trigger_style = False

    # ── Outbreak seeding ────────────────────────────────────────
    seed = ob.seed_by_coverage(camp, timestep=40, coverage=0.05)
    camp.add(seed, first=True)
    purge_campaign_event(seed)

    # ── Health Seeking Behavior ─────────────────────────────────
    adj_slow = probability_per_step_slow / cascade_efficiency
    adj_fast = probability_per_step_fast / cascade_efficiency

    # Low quality care — pre-DOTS
    camp.add(hsb.HSB(
        camp=camp,
        trigger_treatment_list=['TBActivation', 'TBFailedDrugRegimen',
                                'TBTestDefault', 'TBTestNegative', 'TBMDRTestDefault'],
        output_event='TBTestDOTSLow',
        probability_per_step=adj_slow,
        start_day=params.burn_initial,
        duration=params.burn_predots,
        property_restrictions_list=['QualityOfCare:Low'],
        fixed_initial_delay=7
    ))

    # Care seeking — DOTS era onward, with the calibrated secular improvement
    # (cs_gain_per_year).  One block per sub-epoch, both quality streams.
    for _sb in careseeking_schedule(params, cs_gain_per_year):
        for _qoc, _prob in (('Low', adj_slow), ('High', adj_slow * 2)):
            camp.add(hsb.HSB(
                camp=camp,
                trigger_treatment_list=['TBActivation', 'TBFailedDrugRegimen',
                                        'TBTestDefault', 'TBTestNegative',
                                        'TBMDRTestDefault'],
                output_event=f'TBTestDOTS{_qoc}',
                probability_per_step=min(1.0, _prob * _sb.multiplier),
                start_day=_sb.start_day,
                duration=_sb.duration,
                property_restrictions_list=[f'QualityOfCare:{_qoc}'],
                fixed_initial_delay=7
            ))

    # Low quality — treatment seeking after positive test
    camp.add(hsb.HSB(
        camp=camp,
        trigger_treatment_list=['TBTestPositive', 'TBDS_Positive'],
        output_event='TB_Tx',
        probability_per_step=adj_slow,
        start_day=params.burn_initial,
        duration=-1,
        property_restrictions_list=['QualityOfCare:Low'],
        fixed_initial_delay=7
    ))

    # High quality care — pre-DOTS
    camp.add(hsb.HSB(
        camp=camp,
        trigger_treatment_list=['TBActivation', 'TBFailedDrugRegimen',
                                'TBTestDefault', 'TBTestNegative', 'TBMDRTestDefault'],
        output_event='TBTestDOTSHigh',
        probability_per_step=adj_slow * 2,
        start_day=params.burn_initial,
        duration=params.burn_predots,
        property_restrictions_list=['QualityOfCare:High'],
        fixed_initial_delay=7
    ))


    # High quality — treatment seeking after positive test
    camp.add(hsb.HSB(
        camp=camp,
        trigger_treatment_list=['TBTestPositive', 'TBDS_Positive'],
        output_event='TB_Tx',
        probability_per_step=adj_slow * 2,
        start_day=params.burn_initial,
        duration=-1,
        property_restrictions_list=['QualityOfCare:High'],
        fixed_initial_delay=7
    ))

    # Stop testing loop when treated or cleared
    stop_testing = hsb.HSB(
        camp=camp, start_day=1,
        trigger_treatment_list=["TbDrugDistributed", "SymptomaticCleared"],
        output_event="Got_Better_Stop_Testing_Loop",
        probability_per_step=1.0
    )
    stop_testing.Event_Coordinator_Config.Intervention_Config.\
        Actual_IndividualIntervention_Config.Enable_Intervention_Replacement = 1
    camp.add(stop_testing)

    # ── Epoch 1: Pre-DOTS — DiagnosticTreatNeg ──────────────────
    camp.add(dtn.DiagnosticTreatNeg(
        camp=camp, trigger_treatment_list=['TBTestDOTSHigh'],
        base_sensitivity_smearpos=params.sens_smear_pos_pre_GH,
        base_sensitivity_smearneg=params.sens_smear_neg_pre_GH,
        treatment_fraction=0.6, start_day=params.burn_initial,
        duration=params.burn_predots,
        property_restrictions_list=['QualityOfCare:High']
    ))
    camp.add(dtn.DiagnosticTreatNeg(
        camp=camp, trigger_treatment_list=['TBTestDOTSLow'],
        base_sensitivity_smearpos=params.sens_smear_pos_pre_GL,
        base_sensitivity_smearneg=params.sens_smear_neg_pre_GL,
        treatment_fraction=0.6, start_day=params.burn_initial,
        duration=params.burn_predots,
        property_restrictions_list=['QualityOfCare:Low']
    ))

    # Epoch 1 treatment
    camp.add(treat.TBHIVDrugTreatment(
        camp=camp, trigger_treatment_list=['TBTestPositive'],
        drug_name='PreDOTSHigh', latent_multiplier=0,
        start_day=params.burn_initial, duration=params.burn_predots,
        property_restrictions_list=['QualityOfCare:High']
    ))
    camp.add(treat.TBHIVDrugTreatment(
        camp=camp, trigger_treatment_list=['TBTestPositive'],
        drug_name='PreDOTSLow', latent_multiplier=0,
        start_day=params.burn_initial, duration=params.burn_predots,
        property_restrictions_list=['QualityOfCare:Low']
    ))

    # ── Epoch 2 timing constant (diagnostics now come from cascade_schedule) ──
    dots_to_gx = params.genexpert_introduction - params.dots_start

    # Epoch 2 treatment
    for qoc, drug in [('High', 'DOTSHQ'), ('Low', 'DOTSLQ')]:
        for triggers in [['TBTestPositive', 'TBDS_Positive'], ['TB_Tx']]:
            camp.add(treat.TBHIVDrugTreatment(
                camp=camp, trigger_treatment_list=triggers,
                drug_name=drug, latent_multiplier=0,
                start_day=params.dots_start, duration=dots_to_gx,
                property_restrictions_list=[f'QualityOfCare:{qoc}']
            ))
    camp.add(treat.TBHIVDrugTreatment(
        camp=camp, trigger_treatment_list=['TBMDRTestPositive'],
        drug_name='DOTSMDR', latent_multiplier=0,
        start_day=params.dots_start, duration=dots_to_gx,
        property_restrictions_list=[]
    ))

    # ── Epochs 2-3: time-varying cascade schedule (2026-09 trend re-run) ────
    #  Replaces the hand-set DOTS (tf=0.70) / GeneXpert (tf=0.55) steps with a
    #  monotone improvement 1997->2024 driven by tf_gain_per_year, so the model
    #  can reproduce the WHO incidence decline instead of a flat/rising baseline.
    #  The GeneXpert era stays on RampDTN (it is what emits ProviderOrdersTBTest
    #  / ProviderTestNoR, which the resistance diagnostics and the cascade
    #  classification depend on); each sub-epoch continues the overall ramp
    #  rather than restarting it.  See helpers/RECALIBRATION_2026-09.md
    _base_sens_pos = (sens_smear_pos_GXL_override
                      if sens_smear_pos_GXL_override is not None
                      else params.sens_smear_pos_GXL)
    for _blk in cascade_schedule(params, tf_gain_per_year, cs_gain_per_year,
                                 dx_boost=dx_boost, tx_boost=tx_boost,
                                 sens_pos_GXL=_base_sens_pos):
        if _blk.use_ramp:
            camp.add(ramp.RampDTN(
                camp=camp, trigger_treatment_list=[_blk.trigger],
                ramp_time=_blk.duration,
                base_sensitivity_smearpos=_blk.sens_pos_end,
                base_sensitivity_smearneg=_blk.sens_neg_end,
                base_sensitivity_smearpos2=_blk.sens_pos_start,
                base_sensitivity_smearneg2=_blk.sens_neg_start,
                treatment_fraction=_blk.treatment_fraction,
                pos_event='ProviderOrdersTBTest', pos_event2='ProviderTestNoR',
                start_day=_blk.start_day, duration=_blk.duration,
                property_restrictions_list=[f'QualityOfCare:{_blk.qoc}']
            ))
        else:
            camp.add(dtn.DiagnosticTreatNeg(
                camp=camp, trigger_treatment_list=[_blk.trigger],
                base_sensitivity_smearpos=_blk.sens_pos_end,
                base_sensitivity_smearneg=_blk.sens_neg_end,
                treatment_fraction=_blk.treatment_fraction,
                start_day=_blk.start_day, duration=_blk.duration,
                property_restrictions_list=[f'QualityOfCare:{_blk.qoc}']
            ))

    # Resistance diagnostics (two-step MDR confirmation)
    for qoc, sens in [('High', params.sens_resistance_H), ('Low', params.sens_resistance_L)]:
        for trigger in ['ProviderOrdersTBTest', 'ProviderTestNoR']:
            camp.add(rd.ResistanceDiagnostic(
                camp=camp, trigger_treatment_list=[trigger],
                sensitivity=sens, specificity=params.specificity_resistance,
                neg_event='TBDS_Positive',
                treatment_fraction=0.5, treatment_fraction_negative_test=1.0,
                start_day=params.genexpert_introduction, duration=-1,
                property_restrictions_list=[f'QualityOfCare:{qoc}']
            ))

    # Epoch 3 treatment
    for qoc, drug in [('High', 'DOTSHQ'), ('Low', 'DOTSLQ')]:
        for triggers in [['TBTestPositive', 'TBDS_Positive'], ['TB_Tx']]:
            camp.add(treat.TBHIVDrugTreatment(
                camp=camp, trigger_treatment_list=triggers,
                drug_name=drug, latent_multiplier=0,
                start_day=params.genexpert_introduction, duration=-1,
                property_restrictions_list=[f'QualityOfCare:{qoc}']
            ))
    camp.add(treat.TBHIVDrugTreatment(
        camp=camp, trigger_treatment_list=['TBMDRTestPositive'],
        drug_name='DOTSMDR', latent_multiplier=0,
        start_day=params.genexpert_introduction, duration=-1,
        property_restrictions_list=[]
    ))

    # ── ACF (Active Case Finding) — only if coverage > 0 ────────
    if acf_coverage > 0:
        acf_start = params.intervention_day
        acf_prob = min(1.0, acf_coverage)
        camp.add(hsb.HSB(
            camp=camp,
            trigger_treatment_list=['TBActivationPresymptomatic', 'TBActivation'],
            output_event='TBTestDOTSHigh',
            probability_per_step=acf_prob,
            start_day=acf_start, duration=-1,
            property_restrictions_list=['QualityOfCare:High'],
            fixed_initial_delay=1
        ))
        camp.add(hsb.HSB(
            camp=camp,
            trigger_treatment_list=['TBActivationPresymptomatic', 'TBActivation'],
            output_event='TBTestDOTSLow',
            probability_per_step=acf_prob,
            start_day=acf_start, duration=-1,
            property_restrictions_list=['QualityOfCare:Low'],
            fixed_initial_delay=1
        ))

    # ── Secular non-cascade decline in transmission (2026-09-29) ────────
    #  A NodeInfectivityMult event scales total node infectivity along a schedule
    #  running from 1997 to 2024, then holds flat.  It stands for the drivers of
    #  India's historical decline that the care cascade does not represent
    #  (living conditions, nutrition, housing density, demographic change).
    #  Without it the cascade is the only lever that can bend the trajectory, and
    #  the calibration is forced into an implausibly efficient present-day
    #  cascade that leaves no headroom for the Dx and Tx scenarios.
    #  decline = 0 adds nothing, so the previous behaviour is reproduced exactly.
    if infectivity_decline_per_year > 0:
        import emod_api.schema_to_class as _s2c
        _mult_map, _ = infectivity_multiplier_map(params, infectivity_decline_per_year)
        _iv = _s2c.get_class_with_defaults('NodeInfectivityMult', manifest.schema_file)
        _iv.Multiplier_By_Duration = _mult_map
        _ec = _s2c.get_class_with_defaults(
            'StandardInterventionDistributionEventCoordinator', manifest.schema_file)
        _ec.Intervention_Config = _iv
        _ce = _s2c.get_class_with_defaults('CampaignEvent', manifest.schema_file)
        _ce.Event_Coordinator_Config = _ec
        _ce.Start_Day = params.dots_start
        # REQUIRED: Node_Property_Restrictions and Property_Restrictions_Within_Node
        # serialise as {} but EMOD reads them as arrays, which aborts campaign
        # loading with "Bad json_cast" (experiment 2f981103: 1,264 failures).
        # Every other event in this campaign is purged the same way.
        purge_campaign_event(_ce)
        camp.add(_ce)

    # ── Cash (economic tracking) ────────────────────────────────
    if hasattr(cash, 'triggered_cash'):
        cash_on_infection = cash.triggered_cash(
            camp=camp, amount=personal_hc_budget, triggers=['NewInfection'])
        camp.add(cash_on_infection)
    else:
        print('Skipping cash.triggered_cash (not in this emodpy_tbhiv version)')

    return camp


## 6. Demographics Builder

Constructs the multi-node demographics file with age structure, fertility/mortality, HINT transmission groups, and  individual property (High / Low pathway routing).

In [11]:
def build_mi_demog(hint_group_a_bi=3.1, hint_group_b_bi=2.9, hint_group_c_bi=3.5):
    import emodpy_tbhiv.demographics.TBHIVDemographics as Demographics

    demog = Demographics.from_template_node(lat=0, lon=0, pop=initial_pop, forced_id=1)

    demog.AddIndividualPropertyAndHINT(
        Property="QualityOfCare",
        Values=["High", "Low"],
        InitialDistribution=[0.15, 0.85],
        TransmissionMatrix=[[1.0, 1.0], [1.0, 1.0]]
    )
    demog.AddIndividualPropertyAndHINT(
        Property="Geographic",
        Values=["Low", "Mid", "High"],
        InitialDistribution=[0.33, 0.34, 0.33],
        TransmissionMatrix=[
            [hint_group_a_bi, 0.1, 0.1],
            [0.1, hint_group_b_bi, 0.1],
            [0.1, 0.1, hint_group_c_bi]
        ]
    )

    demog.AddAgeDependentTransmission(
        Age_Bin_Edges_In_Years=[0, 20, 40, 60, -1],
        TransmissionMatrix=[
            [3.5, 1.1, 0.5, 0.1],
            [1.1, 2.0, 0.8, 0.1],
            [0.5, 0.8, 1.5, 0.3],
            [0.1, 0.1, 0.3, 0.5]
        ]
    )

    demog.SetMortalityOverTimeFromData(manifest.india_mortality_data, params.base_year)

    return demog


## 7. Sweep Functions

Callbacks for parameter sweeps: random seed, calibration sample mapping, and scenario-level overrides (ACF coverage, cascade efficiency, drug resistance).

In [12]:
def update_sim_random_seed(simulation, value):
    simulation.task.config.parameters.Run_Number = value
    return {"Run_Number": value}

def update_sim_bic(simulation, value):
    simulation.task.config.parameters.Base_Infectivity_Constant = value
    return {"Base_Infectivity_Constant": value}

In [13]:
# 2026-09: single source of truth for which sample entries are CAMPAIGN
# parameters.  Used both by map_sample_to_model_input (calibration sweeps) and
# by sweep_combined (production), so the production run cannot silently drop
# them.  PRODUCTION_CAMP_KWARGS is filled in run_experiment().
PRODUCTION_CAMP_KWARGS = {}


def campaign_kwargs_from_sample(sample):
    """Campaign-affecting entries of a parameter sample, as build_camp kwargs."""
    kw = {}
    if 'Care Seeking Fast Duration Days' in sample:
        kw['probability_per_step_fast'] = 1.0 / sample['Care Seeking Fast Duration Days']
    if 'Care Seeking Slow Duration Days' in sample:
        kw['probability_per_step_slow'] = 1.0 / sample['Care Seeking Slow Duration Days']
    if 'sens_smear_pos_GXL' in sample:
        kw['sens_smear_pos_GXL_override'] = sample['sens_smear_pos_GXL']
    if 'treatment_fraction_LQ' in sample:
        kw['treatment_fraction_LQ_override'] = sample['treatment_fraction_LQ']
    if 'tf_gain_per_year' in sample:
        kw['tf_gain_per_year'] = sample['tf_gain_per_year']
    if 'cs_gain_per_year' in sample:
        kw['cs_gain_per_year'] = sample['cs_gain_per_year']
    if 'infectivity_decline_per_year' in sample:
        kw['infectivity_decline_per_year'] = sample['infectivity_decline_per_year']
    return kw


def map_sample_to_model_input(simulation, sample):
    """Map calibration sample dict to EMOD parameters."""
    sample = dict(sample)  # Work on a copy — .pop() mutates, breaks multi-replicate sweeps

    # 2026-09 FIX: every campaign-affecting parameter is collected here and the
    # campaign callback is registered ONCE at the end.  Previously each of the
    # four campaign parameters called create_campaign_from_callback separately,
    # so they competed: the registration is not additive, and the module-level
    # `camp` object is not reset between calls.  See helpers/RECALIBRATION_2026-09.md
    camp_kwargs = {}

    # v14 fix: only pop if the parameter is actually in the sample
    # (calibration sample contains only the 10 free-calibration parameters,
    # not the full subsample dict used in production runs).
    for p in params.parameters:
        if 'MapTo' in p and p['Name'] in sample:
            simulation.task.config.parameters[p['MapTo']] = sample.pop(p['Name'])

    if 'Care Seeking Fast Duration Days' in sample:
        fast_days = sample.pop('Care Seeking Fast Duration Days')
        camp_kwargs['probability_per_step_fast'] = 1.0 / fast_days

    if 'Care Seeking Slow Duration Days' in sample:
        slow_days = sample.pop('Care Seeking Slow Duration Days')
        camp_kwargs['probability_per_step_slow'] = 1.0 / slow_days

    # v11 wave-2: new calibration parameters wired into build_camp
    if 'sens_smear_pos_GXL' in sample:
        camp_kwargs['sens_smear_pos_GXL_override'] = sample.pop('sens_smear_pos_GXL')

    if 'treatment_fraction_LQ' in sample:
        camp_kwargs['treatment_fraction_LQ_override'] = sample.pop('treatment_fraction_LQ')

    # 2026-09: secular cascade-improvement parameters
    if 'tf_gain_per_year' in sample:
        camp_kwargs['tf_gain_per_year'] = sample.pop('tf_gain_per_year')
    if 'cs_gain_per_year' in sample:
        camp_kwargs['cs_gain_per_year'] = sample.pop('cs_gain_per_year')
    if 'infectivity_decline_per_year' in sample:
        camp_kwargs['infectivity_decline_per_year'] = sample.pop(
            'infectivity_decline_per_year')

    if 'Death Fraction' in sample:
        df = sample.pop('Death Fraction')
        dur = sample.pop('TB Duration', 2.5)
        set_duration_no_hiv(simulation.task, death_fraction=df, duration=dur)
        # v10 Fix A: apply mortality boost AFTER set_duration_no_hiv overwrite
        simulation.task.config.parameters.TB_Active_Mortality_Rate *= 1.5

    if 'TB Presymptomatic Duration Years' in sample:
        val = sample.pop('TB Presymptomatic Duration Years')
        simulation.task.config.parameters.TB_Presymptomatic_Rate = 1.0 / (val * 365)

    if 'TBHIV Duration' in sample:
        sample.pop('TBHIV Duration')

    if 'Relative TB CD4 Infectiousness' in sample:
        val = sample.pop('Relative TB CD4 Infectiousness')
        simulation.task.config.parameters.TB_CD4_Infectiousness = [val] * 7

    if 'CD4_aq_below_200_rel' in sample:
        cd4_below_200 = sample.pop('CD4_aq_below_200_rel')
        cd4_200_300 = sample.pop('CD4_aq_200_300_rel')
        cd4_300_400 = sample.pop('CD4_aq_300_400_rel')
        cd4_400_500 = sample.pop('CD4_aq_400_500_rel')
        cd4_above_500 = sample.pop('CD4_aq_above_500')
        simulation.task.config.parameters.TB_CD4_Activation_Vector = [
            cd4_below_200 * 1e-5, cd4_below_200 * 1e-5,
            cd4_200_300 * 1e-5, cd4_300_400 * 1e-5,
            cd4_400_500 * 1e-5, cd4_above_500 * 1e-5, cd4_above_500 * 1e-5
        ]

    if 'ART Factor' in sample:
        sample.pop('ART Factor')
    if 'Primary HIV multiplier' in sample:
        sample.pop('Primary HIV multiplier')

    # single campaign registration — all campaign parameters applied together
    if camp_kwargs:
        simulation.task.create_campaign_from_callback(
            partial(build_camp, **camp_kwargs))

    return sample


In [14]:
# ── Scenario sweep functions ────────────────────────────────────

def sweep_acf_coverage(simulation, value):
    simulation.task.create_campaign_from_callback(
        partial(build_camp, acf_coverage=value))
    return {"ACF_Coverage": value}

def sweep_cascade_efficiency(simulation, value):
    simulation.task.create_campaign_from_callback(
        partial(build_camp, cascade_efficiency=value))
    return {"Cascade_Efficiency": value}

def sweep_combined(simulation, value):
    # v8: numeric-only tags (cache_experiment_metadata_in_sql calls float()
    # on all tag values; string labels break it). Reconstruct scenario name
    # post-hoc from (acf_coverage, dx_boost, tx_boost).
    acf_cov, dx_boost, tx_boost, label = value
    # 2026-09 FIX: this callback replaces the one registered by
    # map_sample_to_model_input, so it must carry the calibrated campaign
    # parameters too — otherwise tf_gain_per_year / cs_gain_per_year revert to
    # 0 and the production baseline loses the secular cascade improvement.
    simulation.task.create_campaign_from_callback(
        partial(build_camp,
                acf_coverage=acf_cov,
                dx_boost=dx_boost,
                tx_boost=tx_boost,
                **PRODUCTION_CAMP_KWARGS))
    # Scenario_Code: unique 0-7 integer encoding
    #   acf * 4  +  dx * 2  +  tx
    scenario_code = int(bool(acf_cov)) * 4 + int(bool(dx_boost)) * 2 + int(bool(tx_boost))
    return {
        "ACF_Coverage": acf_cov,
        "Dx_Boost": int(dx_boost),
        "Tx_Boost": int(tx_boost),
        "Scenario_Code": scenario_code,
    }

def sweep_calibration_design(simulation, value):
    """v12: Apply a single LHS calibration design point.

    value: dict with keys = parameter names from wave1_design.csv (10 params).
    Uses map_sample_to_model_input pipeline to wire each parameter.

    Returns tag dict with all 10 calibration parameters for reproducibility.
    """
    # Make a working copy so the original sample dict is not mutated
    sample = dict(value)
    # Apply the same mapping pipeline as production subsample
    map_sample_to_model_input(simulation, sample)
    # Tag every parameter so we can match COMPS metadata back to design row
    return {f"cal_{k}": float(v) for k, v in value.items() if k != "design_id"} | \
           {"design_id": int(value.get("design_id", -1))}


## 8. Drug Definitions (for calibration mode)

Defines TBHIV drug regimens (DOTS, MDR, pre-DOTS) with efficacy, duration, and resistance-acquisition parameters used during calibration runs.

In [15]:
def add_drugs_calib(task, resist_pro):
    add_tb_drug_type(task=task, drug_name='DOTSMDR', duration=270,
                     cure_proportion=0.7, death_proportion=0.03,
                     resistance_proportion=0, inactivation_proportion=0.10,
                     relapse_proportion=0.02)
    add_tb_drug_type(task=task, drug_name='PreDOTSHigh', duration=180,
                     cure_proportion=0.7, death_proportion=0.03,
                     resistance_proportion=0, inactivation_proportion=0.10,
                     relapse_proportion=0.02)

## 9. Run Experiment

Assembles the , attaches config/campaign/demographics builders, configures sweeps, and submits the experiment to COMPS.

In [16]:
def run_experiment():
    from idmtools.entities.experiment import Experiment
    from idmtools.builders import SimulationBuilder

    # Create EMODTask
    print('Creating EMODTask...')
    emod_task.dev_mode = False

    task = EMODTask.from_default2(
        eradication_path=manifest.eradication_path,
        campaign_builder=build_camp,
        schema_path=manifest.schema_file,
        param_custom_cb=set_param_fn,
        ep4_path=manifest.ep4_path,
        demog_builder=build_mi_demog,
        config_path='my_config.json'
    )

    print('Adding asset files...')
    task.common_assets.add_directory(assets_directory=manifest.assets_input_dir)

    # Additional config
    update_more_config(task)

    # Bootstrap standard drugs
    import emodpy_tbhiv.standard_drugs as sd
    sd.bootstrap(task.config)

    # Custom drug definitions
    add_tb_drug(task=task, drug_name='ACFDOTS')
    add_tb_drug_type(task=task, drug_name='DOTSMDR', duration=270,
                     cure_proportion=0.7, death_proportion=0.03,
                     resistance_proportion=0, inactivation_proportion=0.10,
                     relapse_proportion=0.02)
    add_tb_drug_type(task=task, drug_name='PreDOTSHigh', duration=180,
                     cure_proportion=0.7, death_proportion=0.03,
                     resistance_proportion=0, inactivation_proportion=0.10,
                     relapse_proportion=0.02)
    add_tb_drug_type(task=task, drug_name='PreDOTSLow', duration=180,
                     cure_proportion=0.6, death_proportion=0.03,
                     resistance_proportion=0, inactivation_proportion=0.10,
                     relapse_proportion=0.02)
    add_tb_drug_type(task=task, drug_name='Universal', duration=90,
                     cure_proportion=0.65, death_proportion=0.03,
                     resistance_proportion=0.02, inactivation_proportion=0.10,
                     relapse_proportion=0.02)

    # Singularity container
    task.set_sif(manifest.sif)

    # ── Build sweep ───────────────────────────────────────────────
    builder = SimulationBuilder()

    # Calibrated subsample — tuned against:
    #   WHO 2020 incidence: 188/100k, NTPS prevalence: 304/100k, WHO mortality: 33/100k
    #
    # Calibration history (year 2020):
    #   fast=0.073, slow=1.03e-5, CS=173/493, DF=0.7, dur=2.5 → inc=265, prev=527, mort=82
    #   fast=0.068, slow=9.4e-6, CS=120/350, DF=0.4, dur=2.0 → inc= 64, prev=117, mort=11
    #
    # Multi-parameter log-linear interpolation at frac=0.758:
    #   inc target: 188, est prev~366→need dur adj, est mort~50→need DF adj
    subsample = {
        # ── Production point: combined wave-1 + wave-2 best fit (wave-2 design 8) ──
        # This is the ORIGINAL subsample with the calibrated values substituted,
        # NOT a bare copy of the design row.  map_sample_to_model_input(subsample)
        # runs in every mode, so each calibration simulation saw this dict first and
        # the design row on top; reproducing the calibration point therefore requires
        # keeping every key the calibration also had.
        'ART Factor': 1.0,
        'Base_Infectivity_Constant': 0.0234460971011517,   # 2026-09 trend calibration (wave-2 design 8)
        'Care Seeking Fast Duration Days': 11.842723428545332,   # 2026-09 trend calibration (wave-2 design 8)
        'Care Seeking Slow Duration Days': 141.0417800875906,   # 2026-09 trend calibration (wave-2 design 8)
        'CD4_aq_200_300_rel': 20.434,
        'CD4_aq_300_400_rel': 3.0,
        'CD4_aq_400_500_rel': 2.0,
        'CD4_aq_above_500': 1.0,
        'CD4_aq_below_200_rel': 54.55,
        'cs_gain_per_year': 0.0351729818161527,   # 2026-09 trend calibration
        'Death Fraction': 0.55,   # with TB Duration above, sets cure and mortality rates; present throughout calibration, so it must stay
        'Post_Infection_Acquisition_Multiplier': 0.4317148260976182,   # 2026-09 trend calibration (wave-2 design 8)
        'Primary HIV multiplier': 1.0,
        'Relative TB CD4 Infectiousness': 0.1,
        'sens_smear_pos_GXL': 0.65,   # 2026-09 trend calibration (wave-2 design 8)
        'TB Duration': 3.5,   # KEPT AT THE CALIBRATION VALUE. The design column is never applied (it needs 'Death Fraction'), so all 300 calibration points ran at this 3.5 y / 0.55 combination, not at the design's duration.
        'TB Presymptomatic Duration Years': 0.3938083619938958,   # 2026-09 trend calibration (wave-2 design 8)
        'TB_Active_Presymptomatic_Infectivity_Multiplier': 0.166,
        'TB_Fast_Progressor_Fraction_Adult': 0.0862685338811378,   # 2026-09 trend calibration (wave-2 design 8)
        'TB_Slow_Progressor_Rate': 1.4576223636924132e-05,   # 2026-09 trend calibration (wave-2 design 8)
        'TBHIV Duration': 1.5,
        'tf_gain_per_year': 0.0038490128965034,   # 2026-09 trend calibration
        'treatment_fraction_LQ': 0.55,   # 2026-09 trend calibration (wave-2 design 8)
    }

    builder.add_sweep_definition(update_sim_random_seed, range(replicates))
    # 2026-09: publish the campaign parameters of the calibrated subsample so
    # sweep_combined can re-apply them (see campaign_kwargs_from_sample)
    global PRODUCTION_CAMP_KWARGS
    PRODUCTION_CAMP_KWARGS = campaign_kwargs_from_sample(subsample)
    print('  campaign kwargs carried into every scenario:',
          sorted(PRODUCTION_CAMP_KWARGS))

    builder.add_sweep_definition(map_sample_to_model_input, [subsample])

    # ── Add scenario-specific sweeps ──────────────────────────────
    if SCENARIO == 'base':
        exp_name = 'TB_India_v5_Base'
    elif SCENARIO == 'acf':
        exp_name = 'TB_India_v5_ACF_Sweep'
        builder.add_sweep_definition(sweep_acf_coverage, ACF_COVERAGE_LEVELS)
    elif SCENARIO == 'cascade':
        exp_name = 'TB_India_v5_Cascade_Efficiency'
        builder.add_sweep_definition(sweep_cascade_efficiency, CASCADE_EFFICIENCY_LEVELS)
    elif SCENARIO == 'combined':
        exp_name = 'TB_India_v5_Combined'
        builder.add_sweep_definition(sweep_combined, COMBINED_SCENARIOS)
    elif SCENARIO in ('wave1_cal', 'wave2_cal'):
        # v12 calibration mode: sweep over LHS design points read from CSV
        import pandas as pd
        wave = '1' if SCENARIO == 'wave1_cal' else '2'
        exp_name = f'TB_India_v12_Wave{wave}_LHS'
        design_df = pd.read_csv(CAL_DESIGN_CSV)
        # Convert each row to a dict and pass as sweep value
        design_points = design_df.to_dict('records')
        print(f'  Calibration sweep: {len(design_points)} design points x '
              f'{replicates} reps = {len(design_points)*replicates} sims')
        builder.add_sweep_definition(sweep_calibration_design, design_points)

    print(f'\nExperiment: {exp_name}')

    experiment = Experiment.from_builder(
        builders=builder, base_task=task, name=exp_name)
    experiment.run(wait_until_done=True, platform=platform)
    task.handle_experiment_completion(experiment)

    print(f'\nExperiment {experiment.uid} succeeded.')

    # Download outputs
    EMODTask.get_file_from_comps(experiment.uid, [
        'InsetChart.json',
        'ReportEventRecorder.csv',
        'Report_TBHIV_ByAge.csv',
        'PropertyReportTB.json',
        'config.json'
    ])
    # v8: cache step wrapped — non-critical, fails on non-numeric tags
    try:
        EMODTask.cache_experiment_metadata_in_sql(experiment.uid)
    except Exception as e:
        print(f'[v8] cache_experiment_metadata_in_sql skipped: {e}')

    return experiment.uid


In [17]:
# Run the experiment
experiment_uid = run_experiment()
print(f'Done. Experiment UID: {experiment_uid}')

Creating EMODTask...
Generating demographics file demographics.json.
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Adding asset files...
  campaign kwargs carried into every scenario: ['cs_gain_per_year', 'probability_per_step_fast', 'probability_per_step_slow', 'sens_smear_pos_GXL_override', 'tf_gain_per_year', 'treatment_fraction_LQ_override']
  Calibration sweep: 100 design points x 25 reps = 2500 sims

Experiment: TB_India_v12_Wave2_LHS

The created experiment can be viewed at https://comps.idmod.org/#explore/Simulations?filters=ExperimentId=496ded25-37bc-f111-92f1-000d3af5294c
Simulations are still being created



Initializing objects for creation: 0simulation [00:00, ?simulation/s]

Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.


Initializing objects for creation:   1%|          | 16/2500 [00:02<06:09,  6.72simulation/s]

Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)
Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.


Initializing objects for creation:   1%|▏         | 32/2500 [00:04<06:11,  6.65simulation/s]

... (309 outputs left out)

Creating Simulations on Comps:   0%|          | 0/2500 [00:00<?, ?simulation/s]               

Use of 'first' flag is deprecated. Use set_schema to start build a new, empty campaign.
Skipping cash.triggered_cash (not in this emodpy_tbhiv version)


Creating Simulations on Comps: 100%|██████████| 2500/2500 [00:02<00:00, 1173.28simulation/s]
Waiting on Experiment TB_India_v12_Wave2_LHS to Finish running: 100%|██████████| 2500/2500 [4:15:43<00:00,  6.14s/simulation]  


Experiment 496ded25-37bc-f111-92f1-000d3af5294c succeeded.

496ded2537bcf11192f1000d3af5294c

Experiment 496ded25-37bc-f111-92f1-000d3af5294c succeeded.
Done. Experiment UID: 496ded25-37bc-f111-92f1-000d3af5294c


---
## 10. Post-Simulation Analysis

After the experiment completes, run the cells below to load simulation outputs (PropertyReport, ReportEventRecorder), compute per-capita time series, and verify calibration targets.

In [18]:
from emodpy.emod_task import EMODTask
import glob

exp_uid = str(globals().get('experiment_uid', '496ded25-37bc-f111-92f1-000d3af5294c'))
print('downloading', exp_uid)

EMODTask.get_file_from_comps(exp_uid, [
    'InsetChart.json', 'ReportEventRecorder.csv',
    'Report_TBHIV_ByAge.csv', 'PropertyReportTB.json', 'config.json',
])
try:
    EMODTask.cache_experiment_metadata_in_sql(exp_uid)
    print('results.db cached.')
except Exception as e:
    print(f'cache failed: {e}')

# 반드시 0이 아닌 숫자가 나와야 함:
print('PropertyReport:', len(glob.glob(f'{exp_uid}/**/PropertyReportTB.json', recursive=True)))
print('RER           :', len(glob.glob(f'{exp_uid}/**/ReportEventRecorder.csv', recursive=True)))

downloading 496ded25-37bc-f111-92f1-000d3af5294c
cache failed: table results already exists
PropertyReport: 2500
RER           : 2500


In [19]:
# ── Load simulation outputs ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import glob, json, os

experiment_uid = '496ded25-37bc-f111-92f1-000d3af5294c'

base_year = 1850  # hardcoded (was: params.base_year)
sim_timestep = 14  # set_config Simulation_Timestep

# ── Try to retroactively download InsetChart.json if missing ──
insetchart_files = glob.glob(f'{experiment_uid}/**/InsetChart.json', recursive=True)
if not insetchart_files:
    print('InsetChart.json not found locally. Attempting retroactive download...')
    try:
        EMODTask.get_file_from_comps(experiment_uid, ['InsetChart.json'])
        insetchart_files = glob.glob(f'{experiment_uid}/**/InsetChart.json', recursive=True)
        print(f'  Downloaded {len(insetchart_files)} InsetChart files.')
    except Exception as e:
        print(f'  Could not download InsetChart: {e}')

# ── 1. Load InsetChart.json (per-capita incidence & prevalence) ──
print(f'Found {len(insetchart_files)} InsetChart files.')
ic_frames = []
for i, f in enumerate(insetchart_files):
    with open(f) as fp:
        ic = json.load(fp)
    header = ic.get('Header', {})
    tstep = header.get('Simulation_Timestep', sim_timestep)
    n_steps = header.get('Timesteps', len(ic['Channels']['Active TB Prevalence']['Data']))
    time_days = np.arange(n_steps) * tstep
    years = base_year + time_days / 365.0

    df_ic = pd.DataFrame({
        'Year': years,
        'Active_TB_Prevalence': ic['Channels']['Active TB Prevalence']['Data'],
        'New_Infections': ic['Channels']['New Infections']['Data'],
        'Statistical_Population': ic['Channels']['Statistical Population']['Data'],
        'sim_idx': i
    })
    df_ic['Incidence_pc'] = df_ic['New_Infections'] / df_ic['Statistical_Population'].replace(0, np.nan)
    df_ic['Prevalence_pc'] = df_ic['Active_TB_Prevalence']
    if 'Disease Deaths' in ic['Channels']:
        df_ic['Disease_Deaths'] = ic['Channels']['Disease Deaths']['Data']
        df_ic['Mortality_pc'] = df_ic['Disease_Deaths'] / df_ic['Statistical_Population'].replace(0, np.nan)
    ic_frames.append(df_ic)

ic_data = pd.concat(ic_frames, ignore_index=True) if ic_frames else None
if ic_data is not None:
    print(f'InsetChart: {len(ic_data):,} rows, {len(ic_frames)} sims, years {ic_data.Year.min():.0f}-{ic_data.Year.max():.0f}')

# ── 2. Load Report_TBHIV_ByAge.csv (fallback for prevalence/incidence) ──
byage_files = glob.glob(f'{experiment_uid}/**/Report_TBHIV_ByAge.csv', recursive=True)
print(f'Found {len(byage_files)} Report_TBHIV_ByAge files.')
byage_data = None
if byage_files:
    byage_frames = []
    for i, f in enumerate(byage_files):
        df_ba = pd.read_csv(f)
        df_ba['sim_idx'] = i
        if 'Year' in df_ba.columns:
            df_ba['CalYear'] = df_ba['Year'] + base_year
        byage_frames.append(df_ba)
    byage_data = pd.concat(byage_frames, ignore_index=True)
    print(f'  Columns: {list(byage_data.columns)[:20]}')
    print(f'  {len(byage_data):,} rows, CalYear {byage_data.CalYear.min():.0f}-{byage_data.CalYear.max():.0f}')

    # Compute per-capita prevalence from ByAge report if InsetChart unavailable
    if ic_data is None and 'Active' in byage_data.columns and 'Population' in byage_data.columns:
        print('  Using Report_TBHIV_ByAge to compute prevalence/incidence...')
        # Sum across age bins per sim per year
        annual = byage_data.groupby(['sim_idx', 'CalYear']).agg(
            Active=('Active', 'sum'),
            Population=('Population', 'sum'),
        ).reset_index()
        annual['Prevalence_pc'] = annual['Active'] / annual['Population'].replace(0, np.nan)
        # NewActiveTB column if available for incidence
        if 'NewActiveTB' in byage_data.columns:
            inc_annual = byage_data.groupby(['sim_idx', 'CalYear'])['NewActiveTB'].sum().reset_index()
            annual = annual.merge(inc_annual, on=['sim_idx', 'CalYear'], how='left')
            annual['Incidence_pc'] = annual['NewActiveTB'] / annual['Population'].replace(0, np.nan)
        # Build ic_data equivalent
        ic_data = annual.rename(columns={'CalYear': 'Year'})
        ic_data['YearInt'] = ic_data['Year'].astype(int)
        print(f'  Reconstructed: {len(ic_data):,} rows')

# ── 3. Load ReportEventRecorder.csv (memory-safe streaming aggregate) ──
# For calibration experiments (2000+ sims), full concat blows up memory.
# Instead, stream per-sim and aggregate to yearly event counts.
rer_files = sorted(glob.glob(f'{experiment_uid}/**/ReportEventRecorder.csv', recursive=True))
print(f'Found {len(rer_files)} ReportEventRecorder files.')
rer_data = None
if rer_files:
    agg_rows = []
    for i, f in enumerate(rer_files):
        try:
            df_re = pd.read_csv(f, usecols=['Time', 'Event_Name'])
        except Exception as e:
            print(f'  skip {f}: {e}'); continue
        df_re['Year'] = (base_year + df_re['Time'] / 365.0).astype(int)
        g = df_re.groupby(['Year', 'Event_Name']).size().reset_index(name='count')
        g['sim_idx'] = i
        g['sim_file'] = f
        agg_rows.append(g)
        if (i + 1) % 200 == 0:
            print(f'  aggregated {i+1}/{len(rer_files)} sims')
    rer_data = pd.concat(agg_rows, ignore_index=True) if agg_rows else None
    if rer_data is not None:
        print(f'  Aggregated: {len(rer_data):,} (sim, year, event) rows, '
              f'{rer_data.sim_idx.nunique()} sims, {rer_data.Event_Name.nunique()} event types')

print('\nData loading complete.')

Found 2500 InsetChart files.
InsetChart: 13,035,000 rows, 2500 sims, years 1850-2050
Found 0 Report_TBHIV_ByAge files.
Found 2500 ReportEventRecorder files.
  aggregated 200/2500 sims
  aggregated 400/2500 sims
  aggregated 600/2500 sims
  aggregated 800/2500 sims
  aggregated 1000/2500 sims
  aggregated 1200/2500 sims
  aggregated 1400/2500 sims
  aggregated 1600/2500 sims
  aggregated 1800/2500 sims
  aggregated 2000/2500 sims
  aggregated 2200/2500 sims
  aggregated 2400/2500 sims
  Aggregated: 8,309,695 (sim, year, event) rows, 2500 sims, 22 event types

Data loading complete.


In [20]:
import glob, os
# ══════════════════════════════════════════════════════════════════════════
# Compute model time series — BEST-FIT design point + replicate CI
# ══════════════════════════════════════════════════════════════════════════
# A calibration experiment holds 200 design points (each a DIFFERENT parameter
# set) x 10 replicates. Overlaying all of them gives the PRIOR-PREDICTIVE spread
# (huge band), NOT calibration uncertainty. The correct calibration figure shows
# the single best-fit design point's trajectory with a CI across its 10
# replicates (Monte-Carlo replicate uncertainty).

import sqlite3
SCALE = 1e5

# ── 1. Map each sim folder (SIM_ID) -> design_id, Run_Number via results.db ──
# ── 1. Map each sim folder (SIM_ID) -> design_id, Run_Number via results.db ──
conn = sqlite3.connect(f'{experiment_uid}/results.db')
tags = pd.read_sql_query("SELECT SIM_ID, design_id, Run_Number FROM results", conn)
sim2design = dict(zip(tags.SIM_ID, tags.design_id))
print(f'results.db: {len(tags)} sims, {tags.design_id.nunique()} design points')

# ── 2. Streaming pass: per-sim annual series (memory-safe; keep only ~60 rows) ──
def sum_pr_channels(pr_channels, base_name, n_steps):
    total = np.zeros(n_steps)
    for ch, val in pr_channels.items():
        if ch.startswith(base_name + ':') or ch == base_name:
            total += np.array(val['Data'])
    return total

pr_files = sorted(glob.glob(f'{experiment_uid}/**/PropertyReportTB.json', recursive=True))
print(f'Found {len(pr_files)} PropertyReport files. Streaming ...')

annual_list = []
for si, pf in enumerate(pr_files):
    sim_id = os.path.basename(os.path.dirname(pf))
    did = sim2design.get(sim_id)
    if did is None:
        continue
    with open(pf, encoding='utf-8') as fp:
        pr = json.load(fp)
    tstep = pr['Header']['Simulation_Timestep']
    n_steps = pr['Header']['Timesteps']
    years_pr = base_year + np.arange(n_steps) * tstep / 365.0

    pop        = sum_pr_channels(pr['Channels'], 'Statistical Population', n_steps)
    active     = sum_pr_channels(pr['Channels'], 'Active TB Infections', n_steps)
    new_active = sum_pr_channels(pr['Channels'], 'New Active Infections', n_steps)
    dd_cum     = sum_pr_channels(pr['Channels'], 'Disease Deaths', n_steps)
    dd_per_ts  = np.diff(dd_cum, prepend=0)
    del pr

    safe_pop = np.maximum(pop, 1)
    df_ts = pd.DataFrame({
        'YearInt': years_pr.astype(int),
        'prev_pc': active / safe_pop,
        'inc_active_pc': new_active / safe_pop,
        'mort_pc': dd_per_ts / safe_pop,
        'pop': pop,
    })
    annual = df_ts.groupby('YearInt').agg(
        prev_pc=('prev_pc', 'mean'),
        inc_active_pc=('inc_active_pc', 'sum'),
        mort_pc=('mort_pc', 'sum'),
        pop=('pop', 'mean'),
    ).reset_index()
    annual['design_id'] = did
    annual['sim_id'] = sim_id
    annual_list.append(annual)
    if (si + 1) % 400 == 0:
        print(f'  {si+1}/{len(pr_files)} sims processed')

allann = pd.concat(annual_list, ignore_index=True)
print(f'Streamed annual frame: {len(allann):,} rows, {allann.design_id.nunique()} design points')

# ── 3. Per design point: 2020 burden (mean across its replicates) ──
TARGETS_BURDEN = {'inc_active_pc': (188, 32.0), 'prev_pc': (304, 36.1), 'mort_pc': (33, 6.0)}
y2020 = allann[allann.YearInt == 2020].copy()
for c in ['inc_active_pc', 'prev_pc', 'mort_pc']:
    y2020[c] = y2020[c] * SCALE
dp = y2020.groupby('design_id').agg(
    inc=('inc_active_pc', 'mean'), prev=('prev_pc', 'mean'), mort=('mort_pc', 'mean'),
).reset_index()

# Implausibility per burden target, then take the max (history-matching rule)
def impl(col, tgt, sig): return (dp[col] - tgt).abs() / sig
dp['I_inc']  = impl('inc',  *TARGETS_BURDEN['inc_active_pc'])
dp['I_prev'] = impl('prev', *TARGETS_BURDEN['prev_pc'])
dp['I_mort'] = impl('mort', *TARGETS_BURDEN['mort_pc'])
dp['I_max']  = dp[['I_inc', 'I_prev', 'I_mort']].max(axis=1)
dp = dp.sort_values('I_max').reset_index(drop=True)

best_did = int(dp.loc[0, 'design_id'])
print('\n=== Best-fit design point (min max-implausibility on 3 burden targets) ===')
print(dp.head(5).to_string(index=False))
print(f'\nBest design_id = {best_did}')
print(f'  2020 Incidence : {dp.loc[0,"inc"]:6.0f}  (target 188, I={dp.loc[0,"I_inc"]:.2f})')
print(f'  2020 Prevalence: {dp.loc[0,"prev"]:6.0f}  (target 304, I={dp.loc[0,"I_prev"]:.2f})')
print(f'  2020 Mortality : {dp.loc[0,"mort"]:6.1f}  (target  33, I={dp.loc[0,"I_mort"]:.2f})')

# print its calibrated parameters
bestrow = tags[tags.design_id == best_did].iloc[0]
print('\nCalibrated parameters at best-fit point:')
for c in tags.columns:
    if c.startswith('cal_'):
        print(f'  {c[4:]:42s} = {bestrow[c]}')

# ── 4. model_ts = mean + 95% CI ACROSS THE 10 REPLICATES of the best point ──
best = allann[allann.design_id == best_did].copy()
best['inc_active_pc'] *= SCALE
best['prev_pc'] *= SCALE
best['mort_pc'] *= SCALE
n_reps = best.sim_id.nunique()
print(f'\nBest-fit point has {n_reps} replicates -> CI reflects replicate uncertainty.')

model_ts = best.groupby('YearInt').agg(
    inc_mean=('inc_active_pc', 'mean'),
    inc_lo=('inc_active_pc', lambda x: x.quantile(0.025)),
    inc_hi=('inc_active_pc', lambda x: x.quantile(0.975)),
    prev_mean=('prev_pc', 'mean'),
    prev_lo=('prev_pc', lambda x: x.quantile(0.025)),
    prev_hi=('prev_pc', lambda x: x.quantile(0.975)),
    mort_mean=('mort_pc', 'mean'),
    mort_lo=('mort_pc', lambda x: x.quantile(0.025)),
    mort_hi=('mort_pc', lambda x: x.quantile(0.975)),
)
# 'inf_mean' kept for backward-compat with the plotting cell (== incidence here)
model_ts['inf_mean'] = model_ts['inc_mean']
model_ts = model_ts.loc[model_ts.index >= 1990]
has_mortality = True

print(f'\nModel time series (best-fit): {len(model_ts)} years')
if 2020 in model_ts.index:
    r = model_ts.loc[2020]
    print(f'  Incidence 2020 : {r["inc_mean"]:6.0f} [{r["inc_lo"]:.0f}-{r["inc_hi"]:.0f}] /100k  (target 188)')
    print(f'  Prevalence 2020: {r["prev_mean"]:6.0f} [{r["prev_lo"]:.0f}-{r["prev_hi"]:.0f}] /100k  (target 304)')
    print(f'  Mortality 2020 : {r["mort_mean"]:6.1f} [{r["mort_lo"]:.1f}-{r["mort_hi"]:.1f}] /100k  (target 33)')


DatabaseError: Execution failed on sql 'SELECT SIM_ID, design_id, Run_Number FROM results': no such column: design_id

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CALIBRATION FIGURE — Model Time Series Overlaid with Empirical Reference Data
# ══════════════════════════════════════════════════════════════════════════════
import os
ref_dir = os.path.join(os.getcwd(), 'data')
os.makedirs('manuscript_figures', exist_ok=True)

SCALE = 1e5   # per capita → per 100,000

# ── Load reference data ──
prev_survey = pd.read_csv(os.path.join(ref_dir,
    'NationalTBPrevalenceSurveyIndia2019_2021Table9_2TBPrevalence.csv'))
india_prev = prev_survey[prev_survey['group'] == 'India'].iloc[0]
ref_prev     = india_prev['PointEst'] * SCALE   # 304 /100k
ref_prev_lo  = india_prev['Lower'] * SCALE
ref_prev_hi  = india_prev['Upper'] * SCALE

chadha = pd.read_csv(os.path.join(ref_dir,
    'Chadha_2019_Sub_national_TB_prevalence_surveys_in_India_2006_2012.csv'))

# Gujarat prevalence survey (age-stratified)
guj_path = os.path.join(ref_dir, 'Gujarat5_Table4.13_BacteriologicalResults_withTable4.5_binConf.csv')
guj = pd.read_csv(guj_path)
guj_total = guj[guj['group'] == 'Total'].iloc[0]  # overall prevalence

# WHO references (per 100k)
ref_inc      = 188.0;  ref_inc_lo  = 140.0;  ref_inc_hi  = 240.0
ref_mort     = 33.0;   ref_mort_lo = 25.0;   ref_mort_hi = 42.0

# ── Compute model time series ──
if model_ts is None:
    print('ERROR: model_ts not computed. Run the time-series computation cell first.')
else:
    yrs = model_ts.index

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # ── Panel A: TB Incidence ──────────────────────────────────────────
    ax = axes[0]
    # Model line + CI
    ax.plot(yrs, model_ts['inc_mean'], '-', color='steelblue', lw=2.5, label='Model', zorder=3)
    ax.fill_between(yrs, model_ts['inc_lo'], model_ts['inc_hi'],
                    alpha=0.25, color='steelblue', label='Model 95% CI', zorder=2)

    # WHO 2020 incidence
    ax.axhspan(ref_inc_lo, ref_inc_hi, alpha=0.10, color='orange', zorder=1)
    ax.axhline(ref_inc, color='darkorange', ls='--', lw=1.5, alpha=0.7, zorder=1)
    ax.errorbar(2020, ref_inc, yerr=[[ref_inc - ref_inc_lo], [ref_inc_hi - ref_inc]],
                fmt='D', color='darkorange', markersize=10, capsize=6, capthick=2, zorder=5,
                label=f'WHO 2020 ({ref_inc:.0f}/100k)')

    ax.axvline(2015, color='gray', ls=':', alpha=0.4)
    ax.axvline(2035, color='red', ls=':', alpha=0.4)
    ax.text(2015.5, ax.get_ylim()[1]*0.95, '2015', fontsize=8, color='gray')
    ax.text(2035.5, ax.get_ylim()[1]*0.95, '2035', fontsize=8, color='red')
    ax.set_title('A. TB Incidence', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Incidence (per 100,000)')
    ax.set_xlim(1990, 2050)
    ax.legend(fontsize=9, loc='upper right', framealpha=0.9)
    ax.grid(axis='y', alpha=0.3)

    # ── Panel B: Active TB Prevalence ──────────────────────────────────
    ax = axes[1]
    # Model line + CI
    ax.plot(yrs, model_ts['prev_mean'], '-', color='teal', lw=2.5, label='Model', zorder=3)
    ax.fill_between(yrs, model_ts['prev_lo'], model_ts['prev_hi'],
                    alpha=0.25, color='teal', label='Model 95% CI', zorder=2)

    # National TB Prevalence Survey 2019-21
    ax.errorbar(2020, ref_prev,
                yerr=[[ref_prev - ref_prev_lo], [ref_prev_hi - ref_prev]],
                fmt='D', color='red', markersize=10, capsize=6, capthick=2, zorder=5,
                label=f'National Survey 2019-21 ({ref_prev:.0f}/100k)')

    # Chadha sub-national surveys 2006-2012
    ax.errorbar(chadha['year'], chadha['mean'] * SCALE,
                yerr=[(chadha['mean'] - chadha['lower']) * SCALE,
                      (chadha['upper'] - chadha['mean']) * SCALE],
                fmt='s', color='purple', markersize=6, capsize=3, alpha=0.8, zorder=4,
                label='Sub-national surveys (Chadha 2019)')

    # Gujarat prevalence survey (~2011-12)
    ax.errorbar(2012, guj_total['PointEst'] * SCALE,
                yerr=[[(guj_total['PointEst'] - guj_total['Lower']) * SCALE],
                      [(guj_total['Upper'] - guj_total['PointEst']) * SCALE]],
                fmt='^', color='green', markersize=8, capsize=4, zorder=4,
                label=f'Gujarat survey ({guj_total["PointEst"]*SCALE:.0f}/100k)')

    ax.set_title('B. Active TB Prevalence', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Prevalence (per 100,000)')
    ax.set_xlim(2000, 2050)
    ax.legend(fontsize=8, loc='upper right', framealpha=0.9)
    ax.grid(axis='y', alpha=0.3)

    # ── Panel C: TB Mortality ──────────────────────────────────────────
    ax = axes[2]
    if has_mortality and 'mort_mean' in model_ts.columns:
        ax.plot(yrs, model_ts['mort_mean'], '-', color='crimson', lw=2.5, label='Model', zorder=3)
        ax.fill_between(yrs, model_ts['mort_lo'], model_ts['mort_hi'],
                        alpha=0.25, color='crimson', label='Model 95% CI', zorder=2)
    elif 'deaths_by_year' in dir() and deaths_by_year is not None and len(deaths_by_year) > 0:
        # Fallback: scale event counts to per 100k using population
        if ic_data is not None and 'Statistical_Population' in ic_data.columns:
            pop_annual = ic_data.groupby(ic_data['Year'].astype(int))['Statistical_Population'].mean()
            sample_rate = 0.05  # Base_Individual_Sample_Rate
            mort_per100k = deaths_by_year / (pop_annual.reindex(deaths_by_year.index) / sample_rate) * SCALE
            mort_per100k = mort_per100k.loc[1990:2050].dropna()
            ax.plot(mort_per100k.index, mort_per100k.values, '-', color='crimson', lw=2.5,
                    label='Model (from events)', zorder=3)
        else:
            ax.plot(deaths_by_year.loc[1990:2050].index,
                    deaths_by_year.loc[1990:2050].values,
                    '-', color='crimson', lw=2.5, label='Model (event counts)', zorder=3)
            ax.set_ylabel('Deaths (count)')

    # WHO mortality reference
    ax.errorbar(2020, ref_mort,
                yerr=[[ref_mort - ref_mort_lo], [ref_mort_hi - ref_mort]],
                fmt='D', color='darkblue', markersize=10, capsize=6, capthick=2, zorder=5,
                label=f'WHO 2020 ({ref_mort:.0f}/100k)')

    ax.axvline(2015, color='gray', ls=':', alpha=0.4)
    ax.axvline(2035, color='red', ls=':', alpha=0.4)
    ax.set_title('C. TB Mortality', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    if ax.get_ylabel() == '':
        ax.set_ylabel('Mortality (per 100,000)')
    ax.set_xlim(1990, 2050)
    ax.legend(fontsize=9, loc='upper right', framealpha=0.9)
    ax.grid(axis='y', alpha=0.3)

    fig.suptitle('Model Calibration vs Empirical Reference Data',
                 fontsize=16, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.savefig('manuscript_figures/calibration_overlay.png', dpi=300, bbox_inches='tight')
    plt.show()

    # ── Calibration summary table ──
    if 2020 in model_ts.index:
        print('\n' + '='*65)
        print('  CALIBRATION SUMMARY (year 2020)')
        print('='*65)
        m_inc = model_ts.loc[2020, 'inc_mean']
        m_prev = model_ts.loc[2020, 'prev_mean']
        print(f'  Incidence:   Model={m_inc:6.0f}   WHO={ref_inc:6.0f}  [{ref_inc_lo:.0f}-{ref_inc_hi:.0f}] /100k')
        print(f'  Prevalence:  Model={m_prev:6.0f}   NTPS={ref_prev:6.0f} [{ref_prev_lo:.0f}-{ref_prev_hi:.0f}] /100k')
        if has_mortality and 'mort_mean' in model_ts.columns:
            m_mort = model_ts.loc[2020, 'mort_mean']
            print(f'  Mortality:   Model={m_mort:6.1f}   WHO={ref_mort:6.0f}  [{ref_mort_lo:.0f}-{ref_mort_hi:.0f}] /100k')
        print('-'*65)
        if 2015 in model_ts.index and 2035 in model_ts.index:
            red_inc = (1 - model_ts.loc[2035, 'inc_mean'] / model_ts.loc[2015, 'inc_mean']) * 100
            print(f'  Incidence  reduction 2015-2035: {red_inc:5.1f}%  (target: 90%)')
            if has_mortality and 'mort_mean' in model_ts.columns:
                red_mort = (1 - model_ts.loc[2035, 'mort_mean'] / model_ts.loc[2015, 'mort_mean']) * 100
                print(f'  Mortality  reduction 2015-2035: {red_mort:5.1f}%  (target: 95%)')
        print('='*65)

    print('\nSaved: manuscript_figures/calibration_overlay.png')

In [ ]:
print('v5 notebook complete.')
print(f'emodpy {emodpy.__version__} | emod-api {emod_api.__version__}')
print('\nRun the figure cells below after experiment outputs are downloaded.')

In [ ]:
# Dump scenario tags to CSV for downstream analysis
# Run this RIGHT AFTER run_experiment() completes (experiment object still in memory)
import os, pandas as pd
from idmtools.entities.experiment import Experiment

# Re-fetch experiment from COMPS to get full sim list with tags
exp = Experiment.from_id(experiment_uid, platform=platform)

SCENARIO_NAMES = {
    0: 'Baseline', 1: 'ACF', 2: 'Dx', 3: 'Tx',
    4: 'ACF+Dx', 5: 'ACF+Tx', 6: 'Dx+Tx', 7: 'ACF+Dx+Tx',
}

rows = []
for sim in exp.simulations:
    tags = dict(sim.tags or {})
    acf = float(tags.get('ACF_Coverage', 0))
    dx  = int(tags.get('Dx_Boost', 0))
    tx  = int(tags.get('Tx_Boost', 0))
    code = int(tags.get('Scenario_Code', int(bool(acf))*4 + dx*2 + tx))
    rows.append({
        'sim_uid': str(sim.uid),
        'Run_Number': int(tags.get('Run_Number', -1)),
        'ACF_Coverage': acf, 'Dx_Boost': dx, 'Tx_Boost': tx,
        'Scenario_Code': code, 'scenario': SCENARIO_NAMES.get(code, f'unknown_{code}'),
    })

df = pd.DataFrame(rows).sort_values(['Scenario_Code', 'Run_Number']).rename(
    columns={'Run_Number': 'rep'})
out_path = os.path.join(str(experiment_uid), 'scenario_tags.csv')
os.makedirs(str(experiment_uid), exist_ok=True)
df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Total sims: {len(df)}')
print(df.groupby('scenario').size())
